<a href="https://colab.research.google.com/github/caoyang0428-art/ML-Assignment/blob/main/Assignment_4_Yang_Cao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 4 - Yang Cao

## Question 3

### Task

Develop a logistic regression classifier for these data. Compare it with the other models that have appeared in this example.

### Preparation

First, we need to load the data and prepare it for the model.

In [1]:
import numpy as np
import pandas as pd
path = 'https://raw.githubusercontent.com/lab30041954/Data/main/'
df = pd.read_csv(path + 'mnist.csv.zip')
print(f"Shape of the loaded DataFrame: {df.shape}")

Shape of the loaded DataFrame: (70000, 785)


### Target vector and features matrix

We separate the labels (target `y`) from the pixel intensities (features `X`). We convert them to NumPy arrays for consistency, as done in class.

In [3]:
y = df['label'].values
X = df.drop(columns='label').values

### Train-test split

We split the data into training and testing sets, allocating 10,000 images for testing, similar to the class example. `random_state=0` ensures reproducibility.

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/7, random_state=0)
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (60000, 784)
Shape of X_test: (10000, 784)
Shape of y_train: (60000,)
Shape of y_test: (10000,)


### Develop a Logistic Regression Classifier


Now, we will train a Logistic Regression Classifier on the prepared data.

In [7]:
from sklearn.linear_model import LogisticRegression
log_reg_clf = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=0)
log_reg_clf.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000, random_state=0)

And then, we evaluate the model.

In [9]:
train_accuracy = log_reg_clf.score(X_train, y_train)
test_accuracy = log_reg_clf.score(X_test, y_test)

print(f"Logistic Regression Training Accuracy: {round(train_accuracy, 3)}")
print(f"Logistic Regression Test Accuracy: {round(test_accuracy, 3)}")

Logistic Regression Training Accuracy: 0.942
Logistic Regression Test Accuracy: 0.912


### Logistic Regression Model Summary and Comparison

The Logistic Regression model achieved the following performance on the MNIST dataset:

* Training Accuracy: 0.942
* Test Accuracy: 0.912
* Overfitting Assessment: Moderate overfitting (3% difference); reasonable generalization.


#### Comparison with other models

**Decision Tree:**
*   Training Accuracy: 0.809
*   Test Accuracy: 0.791
*   Overfitting Assessment: Minimal overfitting; possibly underfitting due to `max_leaf_nodes` constraint.

**Random Forest:**
*   Training Accuracy: 0.924
*   Test Accuracy: 0.916
*   Overfitting Assessment: Slight overfitting; good generalization.

**MLP (Unscaled Data):**
*   Training Accuracy: 0.744
*   Test Accuracy: 0.741
*   Overfitting Assessment: Minimal overfitting; very poor absolute performance.

**MLP (Scaled Data):**
*   Training Accuracy: 0.990
*   Test Accuracy: 0.963
*   Overfitting Assessment: Noticeable overfitting (2.7% difference); excellent absolute performance.

#### Overall Conclusion:

The Logistic Regression model demonstrates strong performance, achieving a test accuracy of 0.912. This places it competitively, slightly below the Random Forest (0.916) and significantly better than the basic Decision Tree (0.791). It also vastly outperforms the unscaled MLP model. While it shows some signs of overfitting (a 3% drop from training to test accuracy), its absolute test performance is respectable, serving as a robust baseline.

The most powerful model, the MLP with scaled data, highlights the critical importance of feature scaling for neural networks. When properly preprocessed, the MLP achieves the highest accuracy (0.963), albeit with a slightly higher degree of overfitting compared to Logistic Regression in this particular setup.

## Question 4

### Task

Calculate a confusion matrix for the logistic regression model (dimension 10x10). Which is the best classified digit? Which is the main source of misclassification?

### Calculate a confusion matrix

In [11]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Make predictions on the test set
y_pred = log_reg_clf.predict(X_test)

# Calculate the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:\n")
print(conf_mat)

Confusion Matrix:

[[ 962    0    2    1    3    8   10    3    6    1]
 [   0 1107    7    3    1    5    2    3   11    2]
 [   9   15  920   21   14    3   14   11   30    3]
 [   2    4   35  893    1   26    1   10   28   13]
 [   3    2    5    2  886    1    9   11    8   35]
 [  12    2    9   28   10  736   19    4   31   12]
 [  10    2    9    0   11   17  934    1    4    1]
 [   5    5   17    5   11    3    1  969    6   42]
 [   5   15    7   23    6   27   10    4  852   14]
 [   5    5    5   10   30    8    1   33    8  864]]


### Analysis of the Confusion Matrix

To identify the best classified digit and the main sources of misclassification, we analyze the confusion matrix:

*   **Best Classified Digit:** This is the digit with the highest value on the main diagonal relative to its row/column sum. A high diagonal value indicates a high number of correct predictions for that class.

*   **Main Source of Misclassification:** This is indicated by large off-diagonal values, where the model frequently predicts one digit when the true digit was another. We look for the largest values outside the main diagonal to find common misclassifications.

In [12]:
# Calculate classification accuracy per digit
class_accuracies = conf_mat.diagonal() / conf_mat.sum(axis=1)

# Best classified digit
best_classified_digit = np.argmax(class_accuracies)
print(f"The best classified digit is: {best_classified_digit} (Accuracy: {class_accuracies[best_classified_digit]:.3f})\n")

# Identify main misclassifications
# Create a copy of the confusion matrix and set diagonal to 0 to easily find misclassifications
misclass_matrix = conf_mat.copy()
np.fill_diagonal(misclass_matrix, 0)

# Find the cell with the highest misclassification count
max_misclass_value = np.max(misclass_matrix)
max_misclass_indices = np.unravel_index(np.argmax(misclass_matrix), misclass_matrix.shape)

true_label = max_misclass_indices[0]
predicted_label = max_misclass_indices[1]

print(f"The most frequent misclassification is when true digit '{true_label}' is predicted as '{predicted_label}' ({max_misclass_value} times).")

The best classified digit is: 1 (Accuracy: 0.970)

The most frequent misclassification is when true digit '7' is predicted as '9' (42 times).


### Summary

The best classified digit is: 1 (Accuracy: 0.970)

The most frequent misclassification is when true digit '7' is predicted as '9' (42 times).

## Question 5

### Task

Try some variations of the MLP model presented in the example of this lecture. For instance, you may increase the number of nodes in the hidden layer to 64, or decrease it to 16. And/or add a second hidden layer. Et cetera.

### Preparation

In [18]:
y = df['label'].values
X = df.drop(columns='label').values
X = X / 255
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/7, random_state=0)

### Variation 1: Increase hidden layer nodes to 64


We'll create a new MLP model with one hidden layer containing 64 nodes, instead of the original 32. We will continue to use the JAX backend and the same `adam` optimizer and `sparse_categorical_crossentropy` loss function as in the example. Importantly, we will use the **rescaled data** (`X_train` and `X_test` are already rescaled from Q7 of the main example) for training, as this significantly improves performance.

In [19]:
import os
os.environ['KERAS_BACKEND'] = 'jax'

from keras import Input, models, layers

# Define the input layer
input_tensor = Input(shape=(784,))

# Define the hidden layer with 64 nodes
x = layers.Dense(64, activation='relu')(input_tensor)

# Define the output layer (same as before)
output_tensor = layers.Dense(10, activation='softmax')(x)

# Instantiate the new MLP model
mlpclf_64nodes = models.Model(input_tensor, output_tensor)

# Compile the model
mlpclf_64nodes.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['acc'])

# Display model summary
mlpclf_64nodes.summary()

# Train the model with rescaled data
print("\nTraining MLP model with 64 hidden nodes...")
mlpclf_64nodes.fit(X_train, y_train, epochs=20, validation_data=(X_test, y_test));
print("Training complete.")

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │        50,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,890 (198.79 KB)

 Trainable params: 50,890 (198.79 KB)

 Non-trainable params: 0 (0.00 B)


Training MLP model with 64 hidden nodes...
Epoch 1/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - acc: 0.8644 - loss: 0.4852 - val_acc: 0.9473 - val_loss: 0.1777
Epoch 2/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - acc: 0.9545 - loss: 0.1559 - val_acc: 0.9581 - val_loss: 0.1396
Epoch 3/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - acc: 0.9680 - loss: 0.1109 - val_acc: 0.9633 - val_loss: 0.1177
Epoch 4/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - acc: 0.9757 - loss: 0.0814 - val_acc: 0.9628 - val_loss: 0.1182
Epoch 5/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - acc: 0.9803 - loss: 0.0658 - val_acc: 0.9703 - val_loss: 0.0982
Epoch 6/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - acc: 0.9836 - loss: 0.0549 - val_acc: 0.9691 - val_loss: 0.1025
Epoch 7/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - acc: 0.9860 - loss: 0.0458 - val_acc: 0.9724 - val_loss: 0.0924
Epoch 8/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - acc: 0.9875 - loss: 0.0399 - val_acc: 0.9722 - val_loss: 0.0

To provide a clear perspective, let's compare this 64-node MLP with the 32-node MLP (on scaled data):


*   **MLP (Scaled Data, 32 Hidden Nodes - from Q7):**
    *   Training Accuracy: 0.9901 (after 20 epochs)
    *   Test Accuracy: 0.9625 (after 20 epochs)
    *   Overfitting Assessment: Noticeable (2.76% difference); excellent absolute performance.

*   **MLP (Scaled Data, 64 Hidden Nodes - current model):**
    *   Training Accuracy: 0.9978 (after 20 epochs)
    *   Test Accuracy: 0.9739 (after 20 epochs)
    *   Overfitting Assessment: Significant (2.39% difference), but highest absolute test accuracy achieved so far.

**Analysis:**

Increasing the number of hidden nodes from 32 to 64 has led to a further improvement in the test accuracy, indicating that the model is fitting the training data very well. The difference between training and test accuracy (~2.39%) indicates that there is still some overfitting, but the absolute test accuracy is the best we've seen yet among the models explored (Decision Tree, Random Forest, Logistic Regression, and 32-node MLP).

This demonstrates that adjusting the neural network's architecture can significantly impact performance, and a slightly larger hidden layer improved generalization for this dataset. Further experimentation with more layers, different numbers of nodes, or regularization techniques could potentially reduce overfitting while maintaining high accuracy.

### Variation 2: Add a second hidden layer


We will now create an MLP model with two hidden layers. For instance, we can keep the first hidden layer with 64 nodes and add a second hidden layer with 32 nodes. We will continue to use the rescaled data, JAX backend, `adam` optimizer, and `sparse_categorical_crossentropy` loss function. This allows us to investigate the impact of increased network depth on performance.

In [20]:
# Set Keras backend (ensure it's set if running independently)
import os
os.environ['KERAS_BACKEND'] = 'jax'

from keras import Input, models, layers

# Define the input layer
input_tensor_two_layers = Input(shape=(784,))

# Define the first hidden layer with 64 nodes
x_two_layers_1 = layers.Dense(64, activation='relu')(input_tensor_two_layers)

# Define the second hidden layer with 32 nodes
x_two_layers_2 = layers.Dense(32, activation='relu')(x_two_layers_1)

# Define the output layer
output_tensor_two_layers = layers.Dense(10, activation='softmax')(x_two_layers_2)

# Instantiate the new MLP model
mlpclf_two_layers = models.Model(input_tensor_two_layers, output_tensor_two_layers)

# Compile the model
mlpclf_two_layers.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['acc'])

# Display model summary
mlpclf_two_layers.summary()

# Train the model with rescaled data
print("\nTraining MLP model with two hidden layers (64, 32 nodes)...")
mlpclf_two_layers.fit(X_train, y_train, epochs=20, validation_data=(X_test, y_test));
print("Training complete.")

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │        50,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,650 (205.66 KB)

 Trainable params: 52,650 (205.66 KB)

 Non-trainable params: 0 (0.00 B)


Training MLP model with two hidden layers (64, 32 nodes)...
Epoch 1/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - acc: 0.8556 - loss: 0.4964 - val_acc: 0.9503 - val_loss: 0.1645
Epoch 2/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - acc: 0.9592 - loss: 0.1411 - val_acc: 0.9600 - val_loss: 0.1301
Epoch 3/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - acc: 0.9707 - loss: 0.0961 - val_acc: 0.9663 - val_loss: 0.1117
Epoch 4/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - acc: 0.9775 - loss: 0.0720 - val_acc: 0.9665 - val_loss: 0.1114
Epoch 5/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - acc: 0.9813 - loss: 0.0588 - val_acc: 0.9687 - val_loss: 0.1018
Epoch 6/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - acc: 0.9857 - loss: 0.0464 - val_acc: 0.9719 - val_loss: 0.1044
Epoch 7/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - acc: 0.9874 - loss: 0.0398 - val_acc: 0.9712 - val_loss: 0.1041
Epoch 8/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - acc: 0.9892 - loss: 0.0348 - val_acc: 0.972

Let's compare the MLP with two hidden layers (64, 32 nodes) to the MLP with a single 64-node hidden layer:

*   **MLP (Scaled Data, 64 Hidden Nodes - Variation 1):**
    *   Training Accuracy: 0.9978 (after 20 epochs)
    *   Test Accuracy: 0.9739 (after 20 epochs)
    *   Overfitting: High training, but good test accuracy. Difference: 2.39%.

*   **MLP (Scaled Data, Two Hidden Layers 64, 32 Nodes - Current Model):**
    *   Training Accuracy: 0.9969 (after 20 epochs)
    *   Test Accuracy: 0.9725 (after 20 epochs)
    *   Overfitting: Similar to the 64-node single layer. Difference: 2.44%.

**Analysis:**

Adding a second hidden layer (64, 32 nodes) did not significantly improve performance compared to the single 64-node layer. Both training and test accuracies were slightly lower (0.9969 vs 0.9978 for training, and 0.9725 vs 0.9739 for test).

The difference between training and test accuracy (overfitting) was also very similar, going from 2.39% to 2.44%. This small change suggests that adding more layers didn't necessarily help the model generalize better; it might have made it slightly harder to train well.


For this dataset, simply adding a second hidden layer didn't make the model better. The single 64-node hidden layer model is still the best performing one so far. This shows that more layers aren't always better, and finding the right model size is important.